# Predator vs Prey (Lotka–Volterra Model)

This notebook implements the LVM-1 predator–prey model:

$$\frac{dp}{dt} = ap - bpP$$
$$\frac{dP}{dt} = \epsilon\,bpP - mP$$

Where:
- $p(t)$ = prey population
- $P(t)$ = predator population
- $a$ = prey per-capita birth rate
- $b$ = interaction constant
- $\epsilon$ = efficiency of converting prey interactions into predator growth
- $m$ = predator mortality rate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Parameters
a = 1.0        # prey growth rate
b = 0.05       # interaction constant
epsilon = 0.5  # predator conversion efficiency
m = 0.6        # predator mortality rate

# Initial populations
p0 = 40.0   # initial prey
P0 = 9.0    # initial predators

# Time settings
t0, tf = 0.0, 60.0
dt = 0.01
t = np.arange(t0, tf + dt, dt)

In [ ]:
def rhs(p, P, a, b, epsilon, m):
    """Right-hand side of LVM-1 equations."""
    dpdt = a * p - b * p * P
    dPdt = epsilon * b * p * P - m * P
    return dpdt, dPdt

# RK4 integration
p = np.zeros_like(t)
P = np.zeros_like(t)
p[0], P[0] = p0, P0

for i in range(len(t) - 1):
    p_i, P_i = p[i], P[i]

    k1p, k1P = rhs(p_i, P_i, a, b, epsilon, m)
    k2p, k2P = rhs(p_i + 0.5 * dt * k1p, P_i + 0.5 * dt * k1P, a, b, epsilon, m)
    k3p, k3P = rhs(p_i + 0.5 * dt * k2p, P_i + 0.5 * dt * k2P, a, b, epsilon, m)
    k4p, k4P = rhs(p_i + dt * k3p, P_i + dt * k3P, a, b, epsilon, m)

    p[i + 1] = p_i + (dt / 6.0) * (k1p + 2 * k2p + 2 * k3p + k4p)
    P[i + 1] = P_i + (dt / 6.0) * (k1P + 2 * k2P + 2 * k3P + k4P)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# Time-series plot
axes[0].plot(t, p, label='Prey p(t)', color='tab:blue', lw=2)
axes[0].plot(t, P, label='Predator P(t)', color='tab:red', lw=2)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Population')
axes[0].set_title('Predator vs Prey Dynamics')
axes[0].grid(alpha=0.3)
axes[0].legend()

# Phase portrait
axes[1].plot(p, P, color='tab:purple', lw=2)
axes[1].set_xlabel('Prey p')
axes[1].set_ylabel('Predator P')
axes[1].set_title('Phase Portrait')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f'Final prey population p(tf): {p[-1]:.3f}')
print(f'Final predator population P(tf): {P[-1]:.3f}')
print('Try changing a, b, epsilon, m, p0, and P0 to explore different dynamics.')

#### JC: Now i know my system is working consistently shown in the book, figure 14.7. In order to prove that it is chaotic to start with is im gonna do a series of phase portraits and dynamics with varying initial conditions.

In [ ]:
# 6 phase portraits: one panel per parameter sweep (a, b, epsilon, m, p0, P0)
def simulate_lotka_volterra(a_, b_, epsilon_, m_, p0_, P0_, t, dt):
    p_run = np.zeros_like(t)
    P_run = np.zeros_like(t)
    p_run[0], P_run[0] = p0_, P0_

    for j in range(len(t) - 1):
        pj, Pj = p_run[j], P_run[j]

        k1p, k1P = rhs(pj, Pj, a_, b_, epsilon_, m_)
        k2p, k2P = rhs(pj + 0.5 * dt * k1p, Pj + 0.5 * dt * k1P, a_, b_, epsilon_, m_)
        k3p, k3P = rhs(pj + 0.5 * dt * k2p, Pj + 0.5 * dt * k2P, a_, b_, epsilon_, m_)
        k4p, k4P = rhs(pj + dt * k3p, Pj + dt * k3P, a_, b_, epsilon_, m_)

        p_run[j + 1] = pj + (dt / 6.0) * (k1p + 2*k2p + 2*k3p + k4p)
        P_run[j + 1] = Pj + (dt / 6.0) * (k1P + 2*k2P + 2*k3P + k4P)

    return p_run, P_run

# Baseline parameter dictionary
base = {"a": a, "b": b, "epsilon": epsilon, "m": m, "p0": p0, "P0": P0}

# Sweep definitions: each entry creates ONE phase portrait panel with multiple overlaid curves
sweeps = [
    ("Vary a", "a", [0.70, 0.90, 1.00, 1.10, 1.30]),
    ("Vary b", "b", [0.70, 0.90, 1.00, 1.10, 1.30]),
    ("Vary epsilon", "epsilon", [0.70, 0.90, 1.00, 1.10, 1.30]),
    ("Vary m", "m", [0.70, 0.90, 1.00, 1.10, 1.30]),
    ("Vary p0", "p0", [0.70, 0.90, 1.00, 1.10, 1.30]),
    ("Vary P0", "P0", [0.70, 0.90, 1.00, 1.10, 1.30]),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, (title, key, factors) in zip(axes, sweeps):
    for fac in factors:
        cfg = base.copy()
        cfg[key] = fac * base[key]
        p_run, P_run = simulate_lotka_volterra(
            cfg["a"], cfg["b"], cfg["epsilon"], cfg["m"], cfg["p0"], cfg["P0"], t, dt
        )

        label = f"{key} = {cfg[key]:.3g}"
        lw = 2.5 if np.isclose(fac, 1.0) else 1.8
        alpha = 1.0 if np.isclose(fac, 1.0) else 0.85
        ax.plot(p_run, P_run, lw=lw, alpha=alpha, label=label)

    ax.set_title(title)
    ax.set_xlabel("Prey p")
    ax.set_ylabel("Predator P")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle("Lotka-Volterra: Six Phase Portraits with Parameter Sweeps", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

JC: This is good, i can potentially see some structure, and its being consistent with what the book tells me and physically how the prey and predator values would change given varying parameters, but i want to see how varying it is. I think longer run time would be ideal. This would make sense that it would have structure and it wouldnt be a fully chaotic system with only 2 degrees of freedom.


In [ ]:
# Expanded sweeps: at least 20 runs per series (using 21 values each)
n_runs = 101
factors = np.linspace(0.6, 1.4, n_runs)  # includes baseline factor = 1.0

# Baseline parameters
base = {"a": a, "b": b, "epsilon": epsilon, "m": m, "p0": p0, "P0": P0}

series = [
    ("Vary a (21 runs)", "a"),
    ("Vary b (21 runs)", "b"),
    ("Vary epsilon (21 runs)", "epsilon"),
    ("Vary m (21 runs)", "m"),
    ("Vary p0 (21 runs)", "p0"),
    ("Vary P0 (21 runs)", "P0"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for ax, (title, key) in zip(axes, series):
    for fac in factors:
        cfg = base.copy()
        cfg[key] = fac * base[key]

        p_run, P_run = simulate_lotka_volterra(
            cfg["a"], cfg["b"], cfg["epsilon"], cfg["m"], cfg["p0"], cfg["P0"], t, dt
        )

        # Light lines for all runs, thicker for baseline
        if np.isclose(fac, 1.0):
            ax.plot(p_run, P_run, color="black", lw=2.8, label="baseline")
        else:
            ax.plot(p_run, P_run, color="tab:blue", alpha=0.22, lw=1.0)

    ax.set_title(title)
    ax.set_xlabel("Prey p")
    ax.set_ylabel("Predator P")
    ax.grid(alpha=0.3)
    ax.legend(loc="best", fontsize=8)

plt.suptitle("Lotka-Volterra Phase Portraits: 21 Overlapped Runs per Parameter", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

print(f"Runs per series: {n_runs} (requirement met: >= 20)")

JC: These loops does not change in strucutre, so i can deduce even after many runs this system is non chaotic. The next step would be adding a degree of freedom that would make it chaotic, like a chance of 

In [ ]:
# Extended ecosystem model: prey (p), predators (P), grass (g), and foliage species (B)
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
# Prey-predator core (from your LVM-1)
a = 3.0          # intrinsic prey growth
b = 0.03         # baseline predation interaction constant
epsilon = 0.25    # predator conversion efficiency from prey
m = 0.9         # predator mortality

# Aging coefficients (all animals)
age_p = 0.015    # prey aging loss rate
age_P = 0.010    # predator aging loss rate
age_B = 0.015    # foliage-species aging loss rate

# Grass dynamics
r_g = 1.8       # grass regrowth rate
K_g = 12000.0      # grass carrying capacity
c_gp = 0.020     # prey grazing pressure on grass (prey lowers grass)
c_gB = 0.010     # foliage-species grazing pressure on grass (B lowers grass)

# Prey gains from grass
eta_gp = 0.020   # efficiency of converting grass intake into prey growth
d_p = 0.06       # baseline natural prey loss

# Escape mechanism: higher grass -> larger escape probability -> lower effective b
k_escape = 0.030  # controls how quickly escape probability grows with grass

# Foliage species (e.g., gopher/badger-like)
r_B = 1       # max growth rate when foliage is abundant
m_B = 0.28       # baseline mortality
H_B = 25.0       # half-saturation for grass-supported B growth
g_thr = 70.0     # grass level where B begins to thrive
s_thr = 0.25     # smoothness of threshold activation

# Predator switches targets when foliage species is present
b_B = 0.030        # predator-foliage interaction constant
epsilon_B = 0.20   # predator conversion efficiency from foliage species
H_switch = 8.0     # controls chance predator targets foliage

# --- Time and initial conditions ---
t0, tf, dt = 0.0, 120.0, 0.01
t = np.arange(t0, tf + dt, dt)

p0 = 240.0   # prey
P0 = 60.0    # predators
g0 = 650.0   # grass
B0 = 12.0    # foliage species (small initial population)

def rhs_extended(p, P, g, B):
    # Smooth threshold for 'new species thrives when grass is large enough'
    trigger_B = 1.0 / (1.0 + np.exp(-s_thr * (g - g_thr)))

    # 1) Grass increases prey escapability -> lower effective b
    escape_prob = 1.0 - np.exp(-k_escape * g)
    b_eff = b * (1.0 - escape_prob)

    # 2) Chance predator switches to foliage species instead of prey
    hunt_foliage_prob = B / (B + H_switch + 1e-12)
    hunt_prey_prob = 1.0 - hunt_foliage_prob

    pred_on_prey = b_eff * hunt_prey_prob * p * P
    pred_on_foliage = b_B * hunt_foliage_prob * B * P

    # Explicit grass-consumption terms
    grass_loss_from_prey = c_gp * g * p
    grass_loss_from_foliage = c_gB * g * B

    # ODEs (include aging losses for all animal populations)
    dpdt = a * p + eta_gp * grass_loss_from_prey - pred_on_prey - d_p * p - age_p * p
    dPdt = epsilon * pred_on_prey + epsilon_B * pred_on_foliage - m * P - age_P * P

    # 3) Both prey and foliage species decrease grass
    dgdt = r_g * g * (1.0 - g / K_g) - grass_loss_from_prey - grass_loss_from_foliage
    dBdt = r_B * B * (g / (g + H_B)) * trigger_B - m_B * B - age_B * B - pred_on_foliage

    return dpdt, dPdt, dgdt, dBdt

# RK4 integration
p = np.zeros_like(t)
P = np.zeros_like(t)
g = np.zeros_like(t)
B = np.zeros_like(t)
p[0], P[0], g[0], B[0] = p0, P0, g0, B0

for i in range(len(t) - 1):
    y1 = rhs_extended(p[i], P[i], g[i], B[i])
    y2 = rhs_extended(
        p[i] + 0.5 * dt * y1[0],
        P[i] + 0.5 * dt * y1[1],
        g[i] + 0.5 * dt * y1[2],
        B[i] + 0.5 * dt * y1[3],
    )
    y3 = rhs_extended(
        p[i] + 0.5 * dt * y2[0],
        P[i] + 0.5 * dt * y2[1],
        g[i] + 0.5 * dt * y2[2],
        B[i] + 0.5 * dt * y2[3],
    )
    y4 = rhs_extended(
        p[i] + dt * y3[0],
        P[i] + dt * y3[1],
        g[i] + dt * y3[2],
        B[i] + dt * y3[3],
    )

    p[i + 1] = p[i] + (dt / 6.0) * (y1[0] + 2*y2[0] + 2*y3[0] + y4[0])
    P[i + 1] = P[i] + (dt / 6.0) * (y1[1] + 2*y2[1] + 2*y3[1] + y4[1])
    g[i + 1] = g[i] + (dt / 6.0) * (y1[2] + 2*y2[2] + 2*y3[2] + y4[2])
    B[i + 1] = B[i] + (dt / 6.0) * (y1[3] + 2*y2[3] + 2*y3[3] + y4[3])

    # Keep populations nonnegative (biological constraint)
    p[i + 1] = max(p[i + 1], 0.0)
    P[i + 1] = max(P[i + 1], 0.0)
    g[i + 1] = max(g[i + 1], 0.0)
    B[i + 1] = max(B[i + 1], 0.0)

# Diagnostics over time
escape_prob_t = 1.0 - np.exp(-k_escape * g)
b_eff_t = b * (1.0 - escape_prob_t)
hunt_foliage_prob_t = B / (B + H_switch + 1e-12)
grass_loss_prey_t = c_gp * g * p
grass_loss_foliage_t = c_gB * g * B

# --- Visualizations ---
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# Time series
axes[0, 0].plot(t, p, label='Prey p(t)', lw=2, color='tab:blue')
axes[0, 0].plot(t, P, label='Predator P(t)', lw=2, color='tab:red')
axes[0, 0].plot(t, g, label='Grass g(t)', lw=2, color='tab:green')
axes[0, 0].plot(t, B, label='Foliage species B(t)', lw=2, color='tab:orange')
axes[0, 0].set_title('Extended Ecosystem Dynamics')
axes[0, 0].set_xlabel('Time')
axes[0, 0].set_ylabel('Population / Biomass')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(alpha=0.3)

# Classic prey-predator phase portrait
axes[0, 1].plot(p, P, color='purple', lw=2)
axes[0, 1].set_title('Phase Portrait: Predator vs Prey')
axes[0, 1].set_xlabel('Prey p')
axes[0, 1].set_ylabel('Predator P')
axes[0, 1].grid(alpha=0.3)

# Grass vs prey
axes[0, 2].plot(g, p, color='teal', lw=2)
axes[0, 2].set_title('Phase Portrait: Prey vs Grass')
axes[0, 2].set_xlabel('Grass g')
axes[0, 2].set_ylabel('Prey p')
axes[0, 2].grid(alpha=0.3)

# Grass vs new species
axes[1, 0].plot(g, B, color='tab:brown', lw=2)
axes[1, 0].axvline(g_thr, color='gray', ls='--', lw=1.5, label='grass threshold')
axes[1, 0].set_title('Phase Portrait: Foliage Species vs Grass')
axes[1, 0].set_xlabel('Grass g')
axes[1, 0].set_ylabel('Foliage Species B')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(alpha=0.3)

# b_eff(t) diagnostic
axes[1, 1].plot(t, b_eff_t, color='black', lw=2, label='b_eff(t)')
axes[1, 1].axhline(b, color='gray', ls='--', lw=1.3, label='baseline b')
axes[1, 1].set_title('Effective Predation Constant b_eff(t)')
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('b_eff')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(alpha=0.3)

# Predator target-switch diagnostic
axes[1, 2].plot(t, hunt_foliage_prob_t, color='tab:olive', lw=2, label='P(predator targets foliage)')
axes[1, 2].set_title('Predator Target Switch Probability')
axes[1, 2].set_xlabel('Time')
axes[1, 2].set_ylabel('Probability')
axes[1, 2].set_ylim(0, 1.02)
axes[1, 2].legend(fontsize=9)
axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final populations -> prey: {p[-1]:.3f}, predators: {P[-1]:.3f}, grass: {g[-1]:.3f}, foliage species: {B[-1]:.3f}')
print(f'b range: baseline={b:.4f}, min b_eff={b_eff_t.min():.4f}, max b_eff={b_eff_t.max():.4f}')
print(f'Predator foliage-target probability range: min={hunt_foliage_prob_t.min():.4f}, max={hunt_foliage_prob_t.max():.4f}')
print(f'Aging coefficients -> prey: {age_p:.3f}, predator: {age_P:.3f}, foliage species: {age_B:.3f}')
print(f'Grass loss rates (max) -> from prey: {grass_loss_prey_t.max():.3f}, from foliage species: {grass_loss_foliage_t.max():.3f}')

JC: ok all of them died, this is not a stable environment. Maybe change the parameters.

In [ ]:
# Adaptive parameter search: wider + self-correcting sweep based on score feedback
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

def safe_sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -60.0, 60.0)))

def simulate_extended(params, tf=140.0, dt=0.03):
    t = np.arange(0.0, tf + dt, dt)
    p = np.zeros_like(t)
    P = np.zeros_like(t)
    g = np.zeros_like(t)
    B = np.zeros_like(t)
    p[0], P[0], g[0], B[0] = params["p0"], params["P0"], params["g0"], params["B0"]

    def rhs_local(pv, Pv, gv, Bv):
        trigger_B = safe_sigmoid(params["s_thr"] * (gv - params["g_thr"]))
        escape_prob = 1.0 - np.exp(-np.clip(params["k_escape"] * gv, 0.0, 60.0))
        b_eff = params["b"] * (1.0 - escape_prob)

        hunt_foliage_prob = Bv / (Bv + params["H_switch"] + 1e-12)
        hunt_prey_prob = 1.0 - hunt_foliage_prob

        pred_on_prey = b_eff * hunt_prey_prob * pv * Pv
        pred_on_foliage = params["b_B"] * hunt_foliage_prob * Bv * Pv

        grass_loss_from_prey = params["c_gp"] * gv * pv
        grass_loss_from_foliage = params["c_gB"] * gv * Bv

        dpdt = (params["a"] * pv + params["eta_gp"] * grass_loss_from_prey
                - pred_on_prey - params["d_p"] * pv - params["age_p"] * pv)
        dPdt = (params["epsilon"] * pred_on_prey + params["epsilon_B"] * pred_on_foliage
                - params["m"] * Pv - params["age_P"] * Pv)
        dgdt = (params["r_g"] * gv * (1.0 - gv / params["K_g"])
                - grass_loss_from_prey - grass_loss_from_foliage)
        dBdt = (params["r_B"] * Bv * (gv / (gv + params["H_B"])) * trigger_B
                - params["m_B"] * Bv - params["age_B"] * Bv - pred_on_foliage)
        return dpdt, dPdt, dgdt, dBdt

    for i in range(len(t) - 1):
        y1 = rhs_local(p[i], P[i], g[i], B[i])
        y2 = rhs_local(p[i] + 0.5 * dt * y1[0], P[i] + 0.5 * dt * y1[1], g[i] + 0.5 * dt * y1[2], B[i] + 0.5 * dt * y1[3])
        y3 = rhs_local(p[i] + 0.5 * dt * y2[0], P[i] + 0.5 * dt * y2[1], g[i] + 0.5 * dt * y2[2], B[i] + 0.5 * dt * y2[3])
        y4 = rhs_local(p[i] + dt * y3[0], P[i] + dt * y3[1], g[i] + dt * y3[2], B[i] + dt * y3[3])

        p[i + 1] = max(p[i] + (dt / 6.0) * (y1[0] + 2 * y2[0] + 2 * y3[0] + y4[0]), 0.0)
        P[i + 1] = max(P[i] + (dt / 6.0) * (y1[1] + 2 * y2[1] + 2 * y3[1] + y4[1]), 0.0)
        g[i + 1] = max(g[i] + (dt / 6.0) * (y1[2] + 2 * y2[2] + 2 * y3[2] + y4[2]), 0.0)
        B[i + 1] = max(B[i] + (dt / 6.0) * (y1[3] + 2 * y2[3] + 2 * y3[3] + y4[3]), 0.0)

        if not np.isfinite(p[i + 1] + P[i + 1] + g[i + 1] + B[i + 1]):
            return None
        if p[i + 1] + P[i + 1] + g[i + 1] + B[i + 1] > 6000:
            return None

    return t, p, P, g, B

def finite_time_lyapunov(params, tf=90.0, dt=0.03, delta=1e-6):
    run1 = simulate_extended(params, tf=tf, dt=dt)
    if run1 is None:
        return np.nan
    t1, p1, P1, g1, B1 = run1

    q = params.copy()
    q["p0"] = max(q["p0"] * (1.0 + delta), 1e-8)
    q["P0"] = max(q["P0"] * (1.0 + delta), 1e-8)
    q["g0"] = max(q["g0"] * (1.0 + delta), 1e-8)
    q["B0"] = max(q["B0"] * (1.0 + delta), 1e-8)

    run2 = simulate_extended(q, tf=tf, dt=dt)
    if run2 is None:
        return np.nan
    t2, p2, P2, g2, B2 = run2

    n = min(len(t1), len(t2))
    tt = t1[:n]
    d = np.sqrt((p2[:n] - p1[:n]) ** 2 + (P2[:n] - P1[:n]) ** 2 + (g2[:n] - g1[:n]) ** 2 + (B2[:n] - B1[:n]) ** 2) + 1e-15
    mask = (tt >= 12.0) & (tt <= 50.0)
    if mask.sum() < 10:
        return np.nan
    return float(np.polyfit(tt[mask], np.log(d[mask]), 1)[0])

# Wider bounds for exploration
PARAM_BOUNDS = {
    "a": (0.4, 3.6),
    "b": (0.008, 0.11),
    "epsilon": (0.05, 0.55),
    "m": (0.25, 1.6),
    "age_p": (0.001, 0.06),
    "age_P": (0.001, 0.06),
    "age_B": (0.001, 0.08),
    "r_g": (0.4, 2.5),
    "c_gp": (0.004, 0.055),
    "c_gB": (0.004, 0.080),
    "d_p": (0.01, 0.25),
    "k_escape": (0.004, 0.120),
    "r_B": (0.05, 1.4),
    "m_B": (0.08, 1.2),
    "b_B": (0.004, 0.10),
    "epsilon_B": (0.03, 0.42),
    "H_switch": (1.5, 30.0),
}

INTEGER_BOUNDS = {
    "p0": (20, 220),
    "P0": (2, 70),
    "g0": (20, 2500),
    "B0": (1, 90),
}

FIXED_PARAMS = {
    "K_g": float(K_g),
    "eta_gp": float(eta_gp),
    "H_B": float(H_B),
    "g_thr": float(g_thr),
    "s_thr": float(s_thr),
}

def clip_param(name, value):
    lo, hi = PARAM_BOUNDS[name]
    return float(np.clip(value, lo, hi))

def clip_int_param(name, value):
    lo, hi = INTEGER_BOUNDS[name]
    return int(np.clip(int(np.round(value)), lo, hi))

def random_candidate():
    cand = {k: rng.uniform(v[0], v[1]) for k, v in PARAM_BOUNDS.items()}
    for k, (lo, hi) in INTEGER_BOUNDS.items():
        cand[k] = int(rng.integers(lo, hi + 1))
    cand.update(FIXED_PARAMS)
    return cand

def mutate_from_elite(elite, sigma):
    cand = {}
    for k, (lo, hi) in PARAM_BOUNDS.items():
        span = hi - lo
        cand[k] = clip_param(k, elite[k] + rng.normal(0.0, sigma * span))

    for k, (lo, hi) in INTEGER_BOUNDS.items():
        span = hi - lo
        cand[k] = clip_int_param(k, elite[k] + rng.normal(0.0, sigma * span))

    cand.update(FIXED_PARAMS)
    return cand

def quick_metrics(run):
    t, p, P, g, B = run
    total = p + P + g + B
    tail = slice(int(0.70 * len(t)), len(t))

    means = np.array([p[tail].mean(), P[tail].mean(), g[tail].mean(), B[tail].mean()])
    mins = np.array([p[tail].min(), P[tail].min(), g[tail].min(), B[tail].min()])
    cvp = np.std(p[tail]) / (np.mean(p[tail]) + 1e-9)
    cvP = np.std(P[tail]) / (np.mean(P[tail]) + 1e-9)
    rel_growth = np.polyfit(t[tail], total[tail], 1)[0] / (np.mean(total[tail]) + 1e-9)
    max_total = float(np.max(total))

    means_main = means[:3]
    mins_main = mins[:3]

    persist = 1.0 / (1.0 + np.exp(-(np.min(means_main) - 4.0) / 2.0))
    nonext = 1.0 / (1.0 + np.exp(-(np.min(mins_main) - 0.03) / 0.03))
    bounded = np.exp(-max(0.0, max_total - 3400.0) / 1000.0)
    oscillatory = np.clip((cvp + cvP) / 0.9, 0.0, 1.3)
    growth = np.exp(-((rel_growth - 0.0015) / 0.014) ** 2)

    qscore = persist * nonext * bounded * oscillatory * growth
    return {
        "qscore": float(qscore),
        "means": means,
        "mins": mins,
        "max_total": max_total,
        "rel_growth": float(rel_growth),
    }

# --- Self-correcting adaptive search ---
n_generations = 12
population_size = 180
elite_keep = 20
explore_ratio_start = 0.45
explore_ratio_end = 0.15
sigma_start = 0.24
sigma_end = 0.06

history = []
archive = []
elites = [random_candidate() for _ in range(elite_keep)]

for gen in range(n_generations):
    phase = gen / max(1, n_generations - 1)
    explore_ratio = (1.0 - phase) * explore_ratio_start + phase * explore_ratio_end
    sigma = (1.0 - phase) * sigma_start + phase * sigma_end

    candidates = []
    n_explore = int(population_size * explore_ratio)
    n_exploit = population_size - n_explore

    # Wide random exploration
    for _ in range(n_explore):
        candidates.append(random_candidate())

    # Self-correcting exploitation around current elites
    for _ in range(n_exploit):
        parent = elites[int(rng.integers(0, len(elites)))]
        candidates.append(mutate_from_elite(parent, sigma=sigma))

    generation_pool = []
    for cand in candidates:
        run = simulate_extended(cand, tf=140.0, dt=0.03)
        if run is None:
            continue
        qm = quick_metrics(run)
        generation_pool.append({"params": cand, "run": run, **qm})

    if len(generation_pool) == 0:
        # Recovery move: reset elites if generation completely fails
        elites = [random_candidate() for _ in range(elite_keep)]
        history.append({"gen": gen + 1, "valid": 0, "best_q": 0.0, "mean_q": 0.0})
        print(f"Generation {gen+1:02d}: valid=0 (reset elites)")
        continue

    generation_pool.sort(key=lambda x: x["qscore"], reverse=True)
    elites = [item["params"] for item in generation_pool[:elite_keep]]
    archive.extend(generation_pool[: max(elite_keep, 25)])

    best_q = generation_pool[0]["qscore"]
    mean_q = float(np.mean([x["qscore"] for x in generation_pool]))
    history.append({"gen": gen + 1, "valid": len(generation_pool), "best_q": best_q, "mean_q": mean_q})

    print(
        f"Generation {gen+1:02d}: valid={len(generation_pool):3d} | "
        f"best_q={best_q:.5f} | mean_q={mean_q:.5f} | sigma={sigma:.3f}"
    )

if len(archive) == 0:
    raise RuntimeError("Adaptive sweep produced no valid trajectories. Increase bounds conservatism or dt.")

# Deduplicate archive (avoid repeatedly rescoring same params)
unique = {}
for item in archive:
    key = tuple(
        [round(item["params"][k], 6) for k in PARAM_BOUNDS.keys()] +
        [int(item["params"][k]) for k in INTEGER_BOUNDS.keys()]
    )
    if key not in unique or item["qscore"] > unique[key]["qscore"]:
        unique[key] = item

pool = list(unique.values())
pool.sort(key=lambda x: x["qscore"], reverse=True)

top_for_lyap = pool[:35]
scored = []
for item in top_for_lyap:
    lyap = finite_time_lyapunov(item["params"], tf=90.0, dt=0.03)
    if not np.isfinite(lyap):
        continue
    chaos = 1.0 / (1.0 + np.exp(-90.0 * (lyap - 0.003)))
    final_score = item["qscore"] * chaos
    scored.append({**item, "lyap": float(lyap), "score": float(final_score)})

if len(scored) == 0:
    raise RuntimeError("No Lyapunov-evaluable candidates found in adaptive sweep.")

scored.sort(key=lambda x: x["score"], reverse=True)
best = scored[0]

print("\nAdaptive sweep summary:")
print(f"Generations: {n_generations}, population/gen: {population_size}")
print(f"Archive size (unique): {len(pool)}")
print(f"Lyapunov-scored finalists: {len(scored)}")

print("\nTop 5 adaptive candidates:")
for i, r in enumerate(scored[:5], start=1):
    mp, mP, mg, mB = r["means"]
    print(
        f"{i}) score={r['score']:.4f}, lyap={r['lyap']:.4f}, rel_growth={r['rel_growth']:.4e}, "
        f"means[p,P,g,B]=[{mp:.2f}, {mP:.2f}, {mg:.2f}, {mB:.2f}], max_total={r['max_total']:.1f}"
    )

print("\nRecommended adaptive balanced-chaotic parameter set:")
for k in [
    "a", "b", "epsilon", "m", "age_p", "age_P", "age_B",
    "r_g", "c_gp", "c_gB", "k_escape", "r_B", "m_B", "b_B", "epsilon_B", "H_switch",
    "p0", "P0", "g0", "B0"
]:
    print(f"{k} = {best['params'][k]:.6g}")

# Plot best candidate
tb, pb, Pb, gb, Bb = simulate_extended(best["params"], tf=220.0, dt=0.02)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].plot(tb, pb, label="Prey p", lw=2)
axes[0].plot(tb, Pb, label="Predator P", lw=2)
axes[0].plot(tb, gb, label="Grass g", lw=2)
axes[0].plot(tb, Bb, label="Foliage B", lw=2)
axes[0].set_title("Best Adaptive Candidate: Time Series")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Population / Biomass")
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(pb, Pb, lw=2, color="purple", label="Predator vs Prey")
axes[1].set_title("Best Adaptive Candidate: Phase Portrait")
axes[1].set_xlabel("Prey p")
axes[1].set_ylabel("Predator P")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

JC: lets test the code to see if its stable.

In [ ]:
# Test run with the provided adaptive parameter set
provided_params = {
    "a": 1.664158,
    "b": 0.0211256,
    "epsilon": 0.549956,
    "m": 1.00727,
    "age_p": 0.0573032,
    "age_P": 0.06,
    "age_B": 0.001,
    "r_g": 2.09402,
    "K_g": float(K_g),
    "c_gp": 0.0312732,
    "c_gB": 0.0285283,
    "eta_gp": float(eta_gp),
    "d_p": float(d_p),
    "k_escape": 0.03729,
    "r_B": 1.09147,
    "m_B": 0.08,
    "H_B": float(H_B),
    "g_thr": float(g_thr),
    "s_thr": float(s_thr),
    "b_B": 0.004,
    "epsilon_B": 0.22297,
    "H_switch": 18.1718,
    "p0": 214,
    "P0": 21,
    "g0": 2000,
    "B0": 1,
}

# Enforce grass growth to be faster than prey growth
provided_params["r_g"] = max(provided_params["r_g"], 1.2 * provided_params["a"])
print(f"Using growth rates -> prey a={provided_params['a']:.4f}, grass r_g={provided_params['r_g']:.4f}")

run_provided = simulate_extended(provided_params, tf=260.0, dt=0.02)
if run_provided is None and 'simulate_manual_unbounded' in globals():
    print('Simulation hit safety bounds in sweep simulator; using fallback integrator.')
    t_prov, p_prov, P_prov, g_prov, B_prov = simulate_manual_unbounded(provided_params, tf=260.0, dt=0.02)
elif run_provided is None:
    raise RuntimeError('Simulation hit safety bounds and no fallback integrator is available.')
else:
    t_prov, p_prov, P_prov, g_prov, B_prov = run_provided

# ---------------------------------------------------------------------
# Food-availability equations (grass-driven coupling for prey + foliage)
# ---------------------------------------------------------------------
# Prey food-limitation factor: F_p(g) = g / (g + H_p)
# Foliage food-limitation factor: F_B(g) = g / (g + H_B)
#
# Food-linked growth terms used for diagnostics:
#   G_p_food(t) = a * p(t) * F_p(g) + eta_gp * c_gp * g(t) * p(t)
#   G_B_food(t) = r_B * B(t) * F_B(g)

H_p = provided_params["H_B"]  # shared half-saturation scale for food response
F_p = g_prov / (g_prov + H_p + 1e-12)
F_B = g_prov / (g_prov + provided_params["H_B"] + 1e-12)

G_p_food = (
    provided_params["a"] * p_prov * F_p
    + provided_params["eta_gp"] * provided_params["c_gp"] * g_prov * p_prov
)
G_B_food = provided_params["r_B"] * B_prov * F_B

print("\nFood-coupled equations added in this cell:")
print("F_p(g) = g / (g + H_p)")
print("F_B(g) = g / (g + H_B)")
print("G_p_food(t) = a*p*F_p(g) + eta_gp*c_gp*g*p")
print("G_B_food(t) = r_B*B*F_B(g)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(t_prov, p_prov, lw=2, label='Prey p')
axes[0].plot(t_prov, P_prov, lw=2, label='Predator P')
axes[0].plot(t_prov, g_prov, lw=2, label='Grass g')
axes[0].plot(t_prov, B_prov, lw=2, label='Foliage B')
axes[0].set_title('Provided Parameters: Time Series')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Population / Biomass')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(p_prov, P_prov, color='purple', lw=2, label='Predator vs Prey')
axes[1].set_title('Provided Parameters: Phase Portrait')
axes[1].set_xlabel('Prey p')
axes[1].set_ylabel('Predator P')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Food-response diagnostics figure
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].plot(t_prov, F_p, lw=2, label='Prey food factor F_p(g)')
axes[0].plot(t_prov, F_B, lw=2, ls='--', label='Foliage food factor F_B(g)')
axes[0].set_title('Food Availability Factors from Grass')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Food factor (0 to 1)')
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(t_prov, G_p_food, lw=2, label='Prey food-linked growth G_p_food')
axes[1].plot(t_prov, G_B_food, lw=2, label='Foliage food-linked growth G_B_food')
axes[1].set_title('Grass-Coupled Growth Terms')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Growth contribution')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

tail = slice(int(0.7 * len(t_prov)), len(t_prov))
print('Final values:')
print(f"p={p_prov[-1]:.3f}, P={P_prov[-1]:.3f}, g={g_prov[-1]:.3f}, B={B_prov[-1]:.3f}")
print('Tail means (last 30%):')
print(f"p={p_prov[tail].mean():.3f}, P={P_prov[tail].mean():.3f}, g={g_prov[tail].mean():.3f}, B={B_prov[tail].mean():.3f}")

### JC: Atleast the prey and predators are still alive, keep changing parameters. I'm gonna ask the agent to do another form of sweeps, using larger values and a larger array to go through, since there are so many parameters doing it by hand would seem like torture>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# COEXISTENCE-BIASED BASELINE PARAMETER REGIME
# ============================================================

manual_params = {
    # prey / predator core
    "a": 1.6,
    "b": 0.018,
    "epsilon": 0.56,
    "m": 0.5,

    # aging / background losses
    "age_p": 0.020,
    "age_P": 0.020,
    "age_B": 0.015,

    # grass
    "r_g": 3.4,
    "K_g": 1800.0,
    "c_gp": 0.00055,
    "c_gB": 0.00090,

    # prey benefit from grass and prey death
    "eta_gp": 0.012,
    "d_p": 0.10,

    # prey escape
    "k_escape": 0.00008,

    # foliage
    "r_B": 0.95,
    "m_B": 0.08,
    "H_B": 30.0,
    "g_thr": 20.0,
    "s_thr": 0.03,

    # predator on foliage
    "b_B": 0.00035,
    "epsilon_B": 0.015,
    "H_switch": 160.0,

    # initial conditions
    "p0": 14.0,
    "P0": 3.0,
    "g0": 900.0,
    "B0": 4.0,
}

# ============================================================
# SIMULATOR
# ============================================================

def simulate_manual(params, tf=260.0, dt=0.02):
    t = np.arange(0.0, tf + dt, dt)
    p = np.zeros_like(t)
    P = np.zeros_like(t)
    g = np.zeros_like(t)
    B = np.zeros_like(t)

    p[0], P[0], g[0], B[0] = params["p0"], params["P0"], params["g0"], params["B0"]

    def rhs_local(pv, Pv, gv, Bv):
        trigger_B = 1.0 / (1.0 + np.exp(-np.clip(params["s_thr"] * (gv - params["g_thr"]), -60.0, 60.0)))

        escape_prob = 1.0 - np.exp(-np.clip(params["k_escape"] * gv, 0.0, 60.0))
        b_eff = params["b"] * (1.0 - escape_prob)

        hunt_foliage_prob = Bv / (Bv + params["H_switch"] + 1e-12)
        hunt_prey_prob = 1.0 - hunt_foliage_prob

        pred_on_prey = b_eff * hunt_prey_prob * pv * Pv
        pred_on_foliage = params["b_B"] * hunt_foliage_prob * Bv * Pv

        grass_loss_from_prey = params["c_gp"] * gv * pv
        grass_loss_from_foliage = params["c_gB"] * gv * Bv

        dpdt = (
            params["a"] * pv
            + params["eta_gp"] * grass_loss_from_prey
            - pred_on_prey
            - params["d_p"] * pv
            - params["age_p"] * pv
        )

        dPdt = (
            params["epsilon"] * pred_on_prey
            + params["epsilon_B"] * pred_on_foliage
            - params["m"] * Pv
            - params["age_P"] * Pv
        )

        dgdt = (
            params["r_g"] * gv * (1.0 - gv / params["K_g"])
            - grass_loss_from_prey
            - grass_loss_from_foliage
        )

        dBdt = (
            params["r_B"] * Bv * (gv / (gv + params["H_B"])) * trigger_B
            - params["m_B"] * Bv
            - params["age_B"] * Bv
            - pred_on_foliage
        )

        return dpdt, dPdt, dgdt, dBdt

    for i in range(len(t) - 1):
        y1 = rhs_local(p[i], P[i], g[i], B[i])
        y2 = rhs_local(
            p[i] + 0.5 * dt * y1[0],
            P[i] + 0.5 * dt * y1[1],
            g[i] + 0.5 * dt * y1[2],
            B[i] + 0.5 * dt * y1[3],
        )
        y3 = rhs_local(
            p[i] + 0.5 * dt * y2[0],
            P[i] + 0.5 * dt * y2[1],
            g[i] + 0.5 * dt * y2[2],
            B[i] + 0.5 * dt * y2[3],
        )
        y4 = rhs_local(
            p[i] + dt * y3[0],
            P[i] + dt * y3[1],
            g[i] + dt * y3[2],
            B[i] + dt * y3[3],
        )

        p[i + 1] = max(p[i] + (dt / 6.0) * (y1[0] + 2 * y2[0] + 2 * y3[0] + y4[0]), 0.0)
        P[i + 1] = max(P[i] + (dt / 6.0) * (y1[1] + 2 * y2[1] + 2 * y3[1] + y4[1]), 0.0)
        g[i + 1] = max(g[i] + (dt / 6.0) * (y1[2] + 2 * y2[2] + 2 * y3[2] + y4[2]), 0.0)
        B[i + 1] = max(B[i] + (dt / 6.0) * (y1[3] + 2 * y2[3] + 2 * y3[3] + y4[3]), 0.0)

        if not np.isfinite(p[i + 1] + P[i + 1] + g[i + 1] + B[i + 1]):
            return None

        if max(p[i + 1], P[i + 1], g[i + 1], B[i + 1]) > 10000:
            return None

    return t, p, P, g, B

# ============================================================
# COEXISTENCE SCORE
# ============================================================

def coexistence_score(t, p, P, g, B):
    arrs = {"p": p, "P": P, "g": g, "B": B}
    maxs = {k: np.max(v) for k, v in arrs.items()}

    tail = slice(int(0.7 * len(t)), len(t))
    means = {k: np.mean(v[tail]) for k, v in arrs.items()}
    mins_tail = {k: np.min(v[tail]) for k, v in arrs.items()}
    stds = {k: np.std(v[tail]) for k, v in arrs.items()}

    # Hard fail if any species nearly disappears in steady-state tail
    if any(v < 1.0 for v in mins_tail.values()):
        return np.inf

    # Hard fail if runaway explosion
    if any(v > 5000.0 for v in maxs.values()):
        return np.inf

    rel_osc = sum(stds[k] / (means[k] + 1e-9) for k in arrs)
    spread = sum((maxs[k] - mins_tail[k]) / (means[k] + 1e-9) for k in arrs)
    final_dev = sum(abs(arrs[k][-1] - means[k]) / (means[k] + 1e-9) for k in arrs)

    return 2.0 * rel_osc + 0.6 * spread + 1.2 * final_dev

# ============================================================
# LARGER SWEEP OF ALL PARAMETERS (2-STAGE)
# ============================================================

SWEEP_BOUNDS = {
    # prey / predator core
    "a": (0.10, 0.35),
    "b": (0.008, 0.030),
    "epsilon": (0.030, 0.100),
    "m": (0.10, 0.35),

    # aging / losses
    "age_p": (0.005, 0.040),
    "age_P": (0.005, 0.040),
    "age_B": (0.005, 0.040),

    # grass
    "r_g": (0.8, 1.8),
    "K_g": (1000.0, 3000.0),
    "c_gp": (0.00020, 0.00120),
    "c_gB": (0.00030, 0.00180),

    # prey food conversion / loss
    "eta_gp": (0.006, 0.022),
    "d_p": (0.06, 0.18),

    # escape
    "k_escape": (0.00002, 0.00020),

    # foliage
    "r_B": (0.55, 1.35),
    "m_B": (0.04, 0.16),
    "H_B": (12.0, 80.0),
    "g_thr": (0.0, 60.0),
    "s_thr": (0.008, 0.080),

    # predator on foliage
    "b_B": (0.00005, 0.00120),
    "epsilon_B": (0.003, 0.040),
    "H_switch": (70.0, 320.0),

    # initial conditions
    "p0": (8, 30),
    "P0": (5, 20),
    "g0": (500, 1300),
    "B0": (5, 100),
}

INT_KEYS = {"p0", "P0", "g0", "B0"}
rng = np.random.default_rng(42)

def sample_all_params(rng_):
    params = {}
    for k, (lo, hi) in SWEEP_BOUNDS.items():
        if k in INT_KEYS:
            params[k] = int(rng_.integers(lo, hi + 1))
        else:
            params[k] = float(rng_.uniform(lo, hi))
    return params

def quick_filter(params):
    if params["r_B"] <= params["m_B"] + params["age_B"]:
        return False
    if params["r_g"] <= 1.02 * params["a"]:
        return False
    return True

# Coarse stage: larger count, shorter and cheaper simulation
N_COARSE_TRIALS = 1800
COARSE_TF = 90.0
COARSE_DT = 0.05
coarse_pool = []

for _ in range(N_COARSE_TRIALS):
    params = sample_all_params(rng)
    if not quick_filter(params):
        continue

    run = simulate_manual(params, tf=COARSE_TF, dt=COARSE_DT)
    if run is None:
        continue

    t, p, P, g, B = run
    score = coexistence_score(t, p, P, g, B)
    if np.isfinite(score):
        coarse_pool.append((score, params))

coarse_pool.sort(key=lambda x: x[0])
TOP_FOR_REFINEMENT = 220
refine_candidates = coarse_pool[:TOP_FOR_REFINEMENT]

print(f"Coarse valid runs: {len(coarse_pool)} / {N_COARSE_TRIALS}")
print(f"Refinement candidates: {len(refine_candidates)}")

# Refined stage: fewer candidates, full simulation horizon
results = []
for coarse_score, params in refine_candidates:
    run = simulate_manual(params, tf=260.0, dt=0.02)
    if run is None:
        continue

    t, p, P, g, B = run
    score = coexistence_score(t, p, P, g, B)
    if np.isfinite(score):
        results.append((score, params, run, coarse_score))

results.sort(key=lambda x: x[0])
print(f"Refined valid coexistence runs: {len(results)} / {len(refine_candidates)}")

if len(results) == 0:
    print("No coexistence solution found in this larger all-parameter sweep.")
    print("Try increasing N_COARSE_TRIALS or broadening bounds.")
else:
    top_n = min(5, len(results))
    print(f"\nTop {top_n} all-parameter sweep results:\n")

    for i in range(top_n):
        score, params, run, coarse_score = results[i]
        t, p, P, g, B = run
        tail = slice(int(0.7 * len(t)), len(t))

        print(f"Rank {i+1} | refined score={score:.4f} | coarse score={coarse_score:.4f}")
        print(params)
        print(
            f"Tail means: p={p[tail].mean():.2f}, P={P[tail].mean():.2f}, "
            f"g={g[tail].mean():.2f}, B={B[tail].mean():.2f}"
        )
        print(
            f"Mins: p={p.min():.2f}, P={P.min():.2f}, g={g.min():.2f}, B={B.min():.2f}"
        )
        print(
            f"Maxs: p={p.max():.2f}, P={P.max():.2f}, g={g.max():.2f}, B={B.max():.2f}"
        )
        print("-" * 90)

    # Plot best result
    best_score, best_params, best_run, _ = results[0]
    t_best, p_best, P_best, g_best, B_best = best_run

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    axes[0].plot(t_best, p_best, lw=2, label='Prey p')
    axes[0].plot(t_best, P_best, lw=2, label='Predator P')
    axes[0].plot(t_best, g_best, lw=2, label='Grass g')
    axes[0].plot(t_best, B_best, lw=2, label='Foliage B')
    axes[0].set_title('Best Larger All-Parameter Sweep: Time Series')
    axes[0].set_xlabel('Time')
    axes[0].set_ylabel('Population / Biomass')
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    axes[1].plot(p_best, P_best, color='purple', lw=2, label='Predator vs Prey')
    axes[1].set_title('Best Larger All-Parameter Sweep: Phase Portrait')
    axes[1].set_xlabel('Prey p')
    axes[1].set_ylabel('Predator P')
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    print("\nBest parameter set to copy:")
    print("manual_params = {")
    for k, v in best_params.items():
        print(f'    "{k}": {repr(v)},')
    print("}")

### JC: This finally works! Im gonna test the bounds of the parameters, and try to display it in 2 different 3 dimensional graphs to see the behavior of the foliage species and grass along with the predator and prey parameters.

In [ ]:
# Single test run using the selected larger-sweep parameter set
mult = 0.0

params_test = {
      
    "a": 0.34534884647541764,
    "b": 0.0298419905605785,
    "epsilon": 0.0848938078040048,
    "m": 0.1405459445740109,
    "age_p": 0.03921983095695926,
    "age_P": 0.016473470505068283,
    "age_B": 0.016676686332239028,
    "r_g": 1.487336952892439,
    "K_g": 1098.0983741489028,
    "c_gp": 0.001025769729907559,
    "c_gB": 0.001795945914731616,
    "eta_gp": 0.01840111644943868,
    "d_p": 0.15380117894487996,
    "k_escape": 9.025646070197918e-05,
    "r_B": 0.6027371112125256,
    "m_B": 0.1351200921081308,
    "H_B": 52.606749909526435,
    "g_thr": 54.560321347325065,
    "s_thr": 0.016349328812011944,
    "b_B": 0.00040089152255822456,
    "epsilon_B": 0.016344277687765297,
    "H_switch": 240.81992713617285,
    "p0": 40*(1 + mult),
    "P0": 30*(1 - mult),
    "g0": 1027*(1 + mult),
    "B0": 97*(1 - mult),
}

run_test_selected = simulate_manual(params_test, tf=800.0, dt=0.02*0.01)
if run_test_selected is None:
    raise RuntimeError("Test run failed (non-finite values or overflow).")

t_s, p_s, P_s, g_s, B_s = run_test_selected

fig, axes = plt.subplots(2,1, figsize=(15, 30))

axes[0].plot(t_s, p_s, lw=2, label='Prey p')
axes[0].plot(t_s, P_s, lw=2, label='Predator P')
axes[0].plot(t_s, g_s, lw=2, label='Grass g')
axes[0].plot(t_s, B_s, lw=2, label='Foliage B')
axes[0].set_title('Selected Parameters: Time Series Test')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Population / Biomass')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].plot(p_s, P_s, color='purple', lw=2, label='Predator vs Prey')
axes[1].set_title('Selected Parameters: Phase Portrait')
axes[1].set_xlabel('Prey p')
axes[1].set_ylabel('Predator P')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# 3D phase portraits requested
fig = plt.figure(figsize=(14, 5.2))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax2 = fig.add_subplot(1, 2, 2, projection='3d')

ax1.plot(p_s, P_s, B_s, color='tab:red', lw=1.6)
ax1.set_title('3D Phase Portrait: Prey-Predator-Foliage')
ax1.set_xlabel('Prey p')
ax1.set_ylabel('Predator P')
ax1.set_zlabel('Foliage B')

ax2.plot(p_s, P_s, g_s, color='tab:green', lw=1.6)
ax2.set_title('3D Phase Portrait: Prey-Predator-Grass')
ax2.set_xlabel('Prey p')
ax2.set_ylabel('Predator P')
ax2.set_zlabel('Grass g')

plt.tight_layout()
plt.show()

tail = slice(int(0.7 * len(t_s)), len(t_s))
print('Final values:')
print(f"p={p_s[-1]:.3f}, P={P_s[-1]:.3f}, g={g_s[-1]:.3f}, B={B_s[-1]:.3f}")
print('Tail means (last 30%):')
print(f"p={p_s[tail].mean():.3f}, P={P_s[tail].mean():.3f}, g={g_s[tail].mean():.3f}, B={B_s[tail].mean():.3f}")
print(f"Coexistence score: {coexistence_score(t_s, p_s, P_s, g_s, B_s):.5f}")

### JC: The plot on the right is a graph of the amount of prey and predators, it starts outside and then spirals in which is consistent of the plot on the left, where it oscillates a lot and then converges into solid values. I tested the limits of the amount of starting values of each component and found that its only converging at a 17% to -32%  difference (positive would be increasing the prey and grass while decreasing the predator and foliage species, and negative values would reverse the signs), but its oscilliatory up until 30% difference. After some thought im gonna animate it to show a large sweep and see at what intervals does it converge.

In [ ]:
# Animated sweep: same plots while mult goes from +0.5 to -0.5
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (needed for 3D projection)

plt.rcParams['animation.embed_limit'] = 100  # MB



# Fallback definitions so this cell can run even after a kernel restart
if 'simulate_manual' not in globals():
    def simulate_manual(params, tf=400.0, dt=0.02):
        t = np.arange(0.0, tf + dt, dt)
        p = np.zeros_like(t)
        P = np.zeros_like(t)
        g = np.zeros_like(t)
        B = np.zeros_like(t)

        p[0], P[0], g[0], B[0] = params["p0"], params["P0"], params["g0"], params["B0"]

        def rhs_local(pv, Pv, gv, Bv):
            trigger_B = 1.0 / (1.0 + np.exp(-np.clip(params["s_thr"] * (gv - params["g_thr"]), -60.0, 60.0)))
            escape_prob = 1.0 - np.exp(-np.clip(params["k_escape"] * gv, 0.0, 60.0))
            b_eff = params["b"] * (1.0 - escape_prob)

            hunt_foliage_prob = Bv / (Bv + params["H_switch"] + 1e-12)
            hunt_prey_prob = 1.0 - hunt_foliage_prob

            pred_on_prey = b_eff * hunt_prey_prob * pv * Pv
            pred_on_foliage = params["b_B"] * hunt_foliage_prob * Bv * Pv

            grass_loss_from_prey = params["c_gp"] * gv * pv
            grass_loss_from_foliage = params["c_gB"] * gv * Bv

            dpdt = (
                params["a"] * pv
                + params["eta_gp"] * grass_loss_from_prey
                - pred_on_prey
                - params["d_p"] * pv
                - params["age_p"] * pv
            )
            dPdt = (
                params["epsilon"] * pred_on_prey
                + params["epsilon_B"] * pred_on_foliage
                - params["m"] * Pv
                - params["age_P"] * Pv
            )
            dgdt = (
                params["r_g"] * gv * (1.0 - gv / params["K_g"])
                - grass_loss_from_prey
                - grass_loss_from_foliage
            )
            dBdt = (
                params["r_B"] * Bv * (gv / (gv + params["H_B"])) * trigger_B
                - params["m_B"] * Bv
                - params["age_B"] * Bv
                - pred_on_foliage
            )
            return dpdt, dPdt, dgdt, dBdt

        for i in range(len(t) - 1):
            y1 = rhs_local(p[i], P[i], g[i], B[i])
            y2 = rhs_local(p[i] + 0.5 * dt * y1[0], P[i] + 0.5 * dt * y1[1], g[i] + 0.5 * dt * y1[2], B[i] + 0.5 * dt * y1[3])
            y3 = rhs_local(p[i] + 0.5 * dt * y2[0], P[i] + 0.5 * dt * y2[1], g[i] + 0.5 * dt * y2[2], B[i] + 0.5 * dt * y2[3])
            y4 = rhs_local(p[i] + dt * y3[0], P[i] + dt * y3[1], g[i] + dt * y3[2], B[i] + dt * y3[3])

            p[i + 1] = max(p[i] + (dt / 6.0) * (y1[0] + 2 * y2[0] + 2 * y3[0] + y4[0]), 0.0)
            P[i + 1] = max(P[i] + (dt / 6.0) * (y1[1] + 2 * y2[1] + 2 * y3[1] + y4[1]), 0.0)
            g[i + 1] = max(g[i] + (dt / 6.0) * (y1[2] + 2 * y2[2] + 2 * y3[2] + y4[2]), 0.0)
            B[i + 1] = max(B[i] + (dt / 6.0) * (y1[3] + 2 * y2[3] + 2 * y3[3] + y4[3]), 0.0)

            if not np.isfinite(p[i + 1] + P[i + 1] + g[i + 1] + B[i + 1]):
                return None
            if max(p[i + 1], P[i + 1], g[i + 1], B[i + 1]) > 10000:
                return None

        return t, p, P, g, B

if 'coexistence_score' not in globals():
    def coexistence_score(t, p, P, g, B):
        arrs = {"p": p, "P": P, "g": g, "B": B}
        maxs = {k: np.max(v) for k, v in arrs.items()}
        tail = slice(int(0.7 * len(t)), len(t))
        means = {k: np.mean(v[tail]) for k, v in arrs.items()}
        mins_tail = {k: np.min(v[tail]) for k, v in arrs.items()}
        stds = {k: np.std(v[tail]) for k, v in arrs.items()}

        if any(v < 1.0 for v in mins_tail.values()) or any(v > 5000.0 for v in maxs.values()):
            return np.inf

        rel_osc = sum(stds[k] / (means[k] + 1e-9) for k in arrs)
        spread = sum((maxs[k] - mins_tail[k]) / (means[k] + 1e-9) for k in arrs)
        final_dev = sum(abs(arrs[k][-1] - means[k]) / (means[k] + 1e-9) for k in arrs)
        return 2.0 * rel_osc + 0.6 * spread + 1.2 * final_dev

# Use the same base parameter set as the previous cell
base_params = {
    "a": 0.34534884647541764,
    "b": 0.0298419905605785,
    "epsilon": 0.0848938078040048,
    "m": 0.1405459445740109,
    "age_p": 0.03921983095695926,
    "age_P": 0.016473470505068283,
    "age_B": 0.016676686332239028,
    "r_g": 1.487336952892439,
    "K_g": 1098.0983741489028,
    "c_gp": 0.001025769729907559,
    "c_gB": 0.001795945914731616,
    "eta_gp": 0.01840111644943868,
    "d_p": 0.15380117894487996,
    "k_escape": 9.025646070197918e-05,
    "r_B": 0.6027371112125256,
    "m_B": 0.1351200921081308,
    "H_B": 52.606749909526435,
    "g_thr": 54.560321347325065,
    "s_thr": 0.016349328812011944,
    "b_B": 0.00040089152255822456,
    "epsilon_B": 0.016344277687765297,
    "H_switch": 240.81992713617285,
}

# --- Sweep controls (change these to sweep any range) ---
mult_start = 0.99
mult_end = -0.99
n_mult = 300
custom_mult_values = None  # e.g., [0.8, 0.3, 0.0, -0.2, -0.6]

if custom_mult_values is not None:
    mult_values = np.array(custom_mult_values, dtype=float)
else:
    if n_mult < 2:
        raise ValueError("n_mult must be at least 2.")
    mult_values = np.linspace(mult_start, mult_end, n_mult)

# Keep runtime manageable for animation (same model, shorter horizon)
tf_anim = 100.0
dt_anim = 0.01
plot_stride = 6

runs = []
for mult in mult_values:
    params_anim = base_params.copy()
    params_anim["p0"] = 40 * (1 + mult)
    params_anim["P0"] = 30 * (1 - mult)
    params_anim["g0"] = 1027 * (1 + mult)
    params_anim["B0"] = 97 * (1 - mult)

    run = simulate_manual(params_anim, tf=tf_anim, dt=dt_anim)
    if run is None:
        runs.append((mult, None, params_anim))
    else:
        t_a, p_a, P_a, g_a, B_a = run
        runs.append((mult, (t_a[::plot_stride], p_a[::plot_stride], P_a[::plot_stride], g_a[::plot_stride], B_a[::plot_stride]), params_anim))

valid_runs = [r for r in runs if r[1] is not None]
if not valid_runs:
    raise RuntimeError("All animation runs failed. Try smaller dt_anim or narrower mult range.")

# Axis limits from all valid runs for stable animation framing
p_max = max(np.max(r[1][1]) for r in valid_runs)
P_max = max(np.max(r[1][2]) for r in valid_runs)
g_max = max(np.max(r[1][3]) for r in valid_runs)
B_max = max(np.max(r[1][4]) for r in valid_runs)
t_max = max(np.max(r[1][0]) for r in valid_runs)

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2)
ax_ts = fig.add_subplot(gs[0, 0])
ax_phase = fig.add_subplot(gs[0, 1])
ax_3d_B = fig.add_subplot(gs[1, 0], projection='3d')
ax_3d_g = fig.add_subplot(gs[1, 1], projection='3d')

line_p, = ax_ts.plot([], [], lw=2, label='Prey p', color='tab:blue')
line_P, = ax_ts.plot([], [], lw=2, label='Predator P', color='tab:red')
line_g, = ax_ts.plot([], [], lw=2, label='Grass g', color='tab:green')
line_B, = ax_ts.plot([], [], lw=2, label='Foliage B', color='tab:orange')

line_phase, = ax_phase.plot([], [], lw=2, color='purple', label='Predator vs Prey')
line_3d_B, = ax_3d_B.plot([], [], [], lw=1.6, color='tab:red')
line_3d_g, = ax_3d_g.plot([], [], [], lw=1.6, color='tab:green')

ax_ts.set_title('Time Series')
ax_ts.set_xlabel('Time')
ax_ts.set_ylabel('Population / Biomass')
ax_ts.set_xlim(0, t_max)
ax_ts.set_ylim(0, 1.05 * max(p_max, P_max, g_max, B_max))
ax_ts.grid(alpha=0.3)
ax_ts.legend(fontsize=8)

ax_phase.set_title('Phase Portrait: Predator vs Prey')
ax_phase.set_xlabel('Prey p')
ax_phase.set_ylabel('Predator P')
ax_phase.set_xlim(0, 1.05 * p_max)
ax_phase.set_ylim(0, 1.05 * P_max)
ax_phase.grid(alpha=0.3)
ax_phase.legend(fontsize=8)

ax_3d_B.set_title('3D: Prey-Predator-Foliage')
ax_3d_B.set_xlabel('Prey p')
ax_3d_B.set_ylabel('Predator P')
ax_3d_B.set_zlabel('Foliage B')
ax_3d_B.set_xlim(0, 1.05 * p_max)
ax_3d_B.set_ylim(0, 1.05 * P_max)
ax_3d_B.set_zlim(0, 1.05 * B_max)

ax_3d_g.set_title('3D: Prey-Predator-Grass')
ax_3d_g.set_xlabel('Prey p')
ax_3d_g.set_ylabel('Predator P')
ax_3d_g.set_zlabel('Grass g')
ax_3d_g.set_xlim(0, 1.05 * p_max)
ax_3d_g.set_ylim(0, 1.05 * P_max)
ax_3d_g.set_zlim(0, 1.05 * g_max)

title = fig.suptitle('', fontsize=13)
text_box = fig.text(0.02, 0.01, '', fontsize=10)

def _set_empty_frame_message(mult):
    line_p.set_data([], [])
    line_P.set_data([], [])
    line_g.set_data([], [])
    line_B.set_data([], [])
    line_phase.set_data([], [])
    line_3d_B.set_data([], [])
    line_3d_B.set_3d_properties([])
    line_3d_g.set_data([], [])
    line_3d_g.set_3d_properties([])
    title.set_text(f'mult = {mult:+.3f} (run failed)')
    text_box.set_text('Simulation unstable for this mult value.')
    return line_p, line_P, line_g, line_B, line_phase, line_3d_B, line_3d_g, title, text_box

def update(frame_idx):
    mult, run_data, _ = runs[frame_idx]
    if run_data is None:
        return _set_empty_frame_message(mult)

    t_a, p_a, P_a, g_a, B_a = run_data

    line_p.set_data(t_a, p_a)
    line_P.set_data(t_a, P_a)
    line_g.set_data(t_a, g_a)
    line_B.set_data(t_a, B_a)

    line_phase.set_data(p_a, P_a)
    line_3d_B.set_data(p_a, P_a)
    line_3d_B.set_3d_properties(B_a)
    line_3d_g.set_data(p_a, P_a)
    line_3d_g.set_3d_properties(g_a)

    tail = slice(int(0.7 * len(t_a)), len(t_a))
    score = coexistence_score(t_a, p_a, P_a, g_a, B_a)
    title.set_text(f'mult sweep frame {frame_idx + 1}/{len(mult_values)} | mult = {mult:+.3f}')
    text_box.set_text(
        f"final: p={p_a[-1]:.2f}, P={P_a[-1]:.2f}, g={g_a[-1]:.2f}, B={B_a[-1]:.2f} | "
        f"tail mean: p={p_a[tail].mean():.2f}, P={P_a[tail].mean():.2f}, g={g_a[tail].mean():.2f}, B={B_a[tail].mean():.2f} | "
        f"score={score:.4f}"
    )

    return line_p, line_P, line_g, line_B, line_phase, line_3d_B, line_3d_g, title, text_box

ani = FuncAnimation(fig, update, frames=len(mult_values), interval=280, blit=False, repeat=True)
plt.close(fig)
display(HTML(ani.to_jshtml()))
print(f"Animation created with {len(mult_values)} frames (mult from {mult_values[0]:+.3f} to {mult_values[-1]:+.3f}).")

In [ ]:
# Bifurcation diagram: mult vs final prey/predator populations
import numpy as np
import matplotlib.pyplot as plt

# Requires simulate_manual(...) defined in a previous cell
if "simulate_manual" not in globals():
    raise RuntimeError("simulate_manual is not defined. Run the cell that defines it first.")

# Fallback base params (used if not already defined)
if "base_params" not in globals():
    base_params = {
        "a": 0.34534884647541764,
        "b": 0.0298419905605785,
        "epsilon": 0.0848938078040048,
        "m": 0.1405459445740109,
        "age_p": 0.03921983095695926,
        "age_P": 0.016473470505068283,
        "age_B": 0.016676686332239028,
        "r_g": 1.487336952892439,
        "K_g": 1098.0983741489028,
        "c_gp": 0.001025769729907559,
        "c_gB": 0.001795945914731616,
        "eta_gp": 0.01840111644943868,
        "d_p": 0.15380117894487996,
        "k_escape": 9.025646070197918e-05,
        "r_B": 0.6027371112125256,
        "m_B": 0.1351200921081308,
        "H_B": 52.606749909526435,
        "g_thr": 54.560321347325065,
        "s_thr": 0.016349328812011944,
        "b_B": 0.00040089152255822456,
        "epsilon_B": 0.016344277687765297,
        "H_switch": 240.81992713617285,
    }

# Sweep settings
mult_values = np.linspace(0.4, -0.4, 1000)
tf_bif, dt_bif = 350.0, 0.03

mult_ok, prey_final, pred_final = [], [], []

for mult in mult_values:
    params = base_params.copy()

    # Your initial-condition mapping
    params["p0"] = max(1e-6, 40 * (1 + mult))
    params["P0"] = max(1e-6, 30 * (1 - mult))
    params["g0"] = max(1e-6, 1027 * (1 + mult))
    params["B0"] = max(1e-6, 97 * (1 - mult))

    run = simulate_manual(params, tf=tf_bif, dt=dt_bif)
    if run is None:
        continue

    _, p, P, _, _ = run
    mult_ok.append(mult)
    prey_final.append(p[-1])
    pred_final.append(P[-1])

if len(mult_ok) == 0:
    raise RuntimeError("No valid runs found for this sweep.")

# Plot
fig, ax = plt.subplots(figsize=(10.5, 5.5))
ax.scatter(mult_ok, prey_final, s=14, alpha=0.65, color="tab:blue", label="Final prey p(tf)")
ax.scatter(mult_ok, pred_final, s=14, alpha=0.65, color="tab:red", label="Final predator P(tf)")
ax.set_title("Bifurcation Diagram: mult vs Final Population")
ax.set_xlabel("mult")
ax.set_ylabel("Final population at tf")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Valid points: {len(mult_ok)} / {len(mult_values)}")
print(f"mult range sampled: {mult_values[0]:+.3f} to {mult_values[-1]:+.3f}")